# Phase 2 (L-NER) — Run on Phase-1 **Decoder-only model 03** (ALiBi + Multi-Head Latent Attention), with **partial fine-tuning**

Config-driven, like the other two NER notebooks: edit **Cell 2** only.

### What this architecture means for NER
| | Decoder 01 | Decoder 02 | **Decoder 03 (this notebook)** |
|---|---|---|---|
| positions | none (NoPE) | RoPE | **ALiBi**: a fixed bias `−m_h·(i−j)` added to the attention logits, no position parameters |
| attention | MHA | local/global + GQA | **MLA**: K and V of every head are rebuilt from a 128-dim latent `c_t` |
| layers | 12 | 12 | **13** |
| can it be unmasked? | no | yes | **yes**: ALiBi gives distance information through the *bias*, so a symmetric version `−m_h·|i−j|` is well defined |

### `METHOD`
| `METHOD` | What is trained | Notes |
|---|---|---|
| `"partial"` | the **top `PARTIAL_K` blocks** + final RMSNorm + head | **default** (run #4 of your plan). Embeddings and the lower blocks are frozen |
| `"full"` | everything | smoke test / reference |
| `"lora"` / `"lorma"` | adapters on all 10 Linear types (incl. MLA's `kv_down`, `k_up`, `v_up`) + head | same code as the Decoder-02 notebook |
| `"norm_only"` | all RMSNorm gains (incl. MLA's latent norm) + head | no biases exist, so BitFit is impossible |

**Why partial fine-tuning works (Modules 2.2 and 2.4).** Lower layers of a pretrained Transformer compute general lexical and syntactic features; upper layers are closer to the output and more task-specific. With a residual stream, gradients also reach the upper blocks without passing through the lower ones, so training only the top blocks is a stable, cheap form of transfer learning (the middle ground between "freeze everything, train a linear head" = `PARTIAL_K=0`, and full fine-tuning). Autograd does not even store activations for the frozen part, so it also uses less memory and time.

### `ATTN_MODE`: the design decision for this notebook
| mode | what the attention looks like | use with |
|---|---|---|
| `"bidir_top"` | blocks `0 … n−K−1` stay **causal** (exactly the Phase-1 function); the top `BIDIR_TOP_K` blocks use **symmetric ALiBi** (bidirectional) | **default; the natural partner of `partial`** |
| `"bidir"` | all 13 blocks bidirectional | `full` / `lora` / `lorma` |
| `"echo"` | `[BOS] w [EOS][BOS] w`, tag the 2nd copy | possible; ALiBi penalises the distance to the 1st copy, so only the small-slope heads can reach it |
| `"causal"` | left context only | baseline |

Why `bidir_top` and not `bidir` for partial fine-tuning: with partial fine-tuning the lower blocks are frozen. If we unmask *every* block, those frozen blocks suddenly see a different attention pattern than the one they were pretrained for, and nothing is allowed to adapt to it. Keeping them causal means they run exactly as in Phase 1. Each token's causal representation already summarises its left context, so the unmasked top blocks can mix these summaries in both directions and still bring in right context. (This is a choice of mine, not something from a lecture; the `bidir` vs `bidir_top` comparison is a one-row ablation for your report.)

### Pipeline (identical to the other NER notebooks)
document → BPE tokens with character offsets → 25 BIO tags → sliding windows → Phase-1 backbone → `Linear(512→25)` → average overlapping windows → BIO decode (+ learned edge-punctuation trim) → character spans → strict macro-F1.

### Outputs (in `WORK_DIR/runs/<RUN_NAME>/`)
`run_log.json`, `ner_*.pt` (trainable weights only), `submission/submission.zip`.

In [ ]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

In [ ]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "dec03_partial_smoke"
METHOD     = "partial"        # "partial" | "full" | "lora" | "lorma" | "norm_only"
SMOKE_TEST = True             # True: ~250 steps on a subset, dev check only, NO submission written
ATTN_MODE  = "bidir_top"      # "bidir_top" | "bidir" | "echo" | "causal"   (see the first markdown cell)
PARTIAL_K   = 4               # METHOD="partial": train the top K blocks (0 = train only final norm + head, i.e. a linear probe)
BIDIR_TOP_K = 4               # ATTN_MODE="bidir_top": the top K blocks become bidirectional (keep equal to PARTIAL_K for run #4)

# ---- Phase-1 checkpoint (written by caps_decoder_only_architecture_03.ipynb: runs/<ARCH>/best.pt) ---------------------------------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-dec-03-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data ---------------------------------------------------------------------------------
HF_REPO = "Exploration-Lab/CS-781-Capstone"      # same repo as Phase 1; files are Phase_2_LNER/{Train,Test}/...
LOCAL_DATA_DIR = None                             # or a folder that contains Phase_2_LNER/ (checked first)
TRAIN_FOLDS = [1]
VAL_FOLDS   = [2]                                 # [] => no validation (use for the final fit on both folds)

# ---- Training -----------------------------------------------------------------------------
EPOCHS      = 5
BATCH       = 8                                   # sequences per micro-batch
ACCUM       = 2                                   # gradient accumulation (effective batch = BATCH*ACCUM)
WARMUP_FRAC = 0.06
GRAD_CLIP   = 1.0
HP = {                                            # per-method hyper-parameters (starting points, tune on dev)
    "partial":   dict(lr=2e-4, head_lr=1e-3, wd=0.01),
    "full":      dict(lr=1e-4, head_lr=1e-3, wd=0.01),
    "lora":      dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, dropout=0.05),
    "lorma":     dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, variant="plus"),   # "plus" (LoRMA+) | "pi" (LoRMA_pi)
    "norm_only": dict(lr=1e-3, head_lr=1e-3, wd=0.0),
}
# every Linear name in this architecture: MHA/GQA-style (q,k,v,o), MLA (q, kv_down, k_up, v_up, o) and the SwiGLU FFN
ADAPTER_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "kv_down", "k_up", "v_up", "gate_proj", "up_proj", "down_proj")
HEAD_DROPOUT = 0.1
TRIM_PUNCT   = True                               # learn edge punctuation that never starts/ends a gold entity; trim it when decoding

# ---- Windows / class imbalance --------------------------------------------------------------
WINDOW_OVERRIDE, STRIDE_OVERRIDE = None, None     # tokens per window / stride; None => defaults below
NEG_WINDOW_KEEP = 0.3                             # keep this fraction of windows that contain no entity (per epoch)
CLASS_WEIGHT    = "sqrt_inv"                      # "sqrt_inv" | "none"
MAX_CLASS_W     = 10.0
O_SCALE_GRID    = (0.25, 0.4, 0.6, 0.8, 1.0, 1.25, 1.6, 2.0, 3.0)   # post-hoc scaling of P(O), tuned on dev
O_SCALE_FINAL   = 1.0                             # used when VAL_FOLDS=[] (copy the tuned value from the dev run)

# ---- Smoke test -----------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_VAL_DOCS = 250, 25

# ---- Knowledge distillation (run #6): let this run serve as the TEACHER -------------------------------------------------------
EXPORT_SOFT_LABELS = False        # True (full run, SMOKE_TEST=False): save this model's per-token distributions for the KD notebook
EXPORT_FOLDS = [1, 2]             # folds to export (the student can only be distilled on folds that were exported)

# ---- derived ----------------------------------------------------------------------------------
assert METHOD in HP and ATTN_MODE in ("causal", "echo", "bidir", "bidir_top")
if ATTN_MODE == "echo":
    WINDOW, STRIDE = 400, 200                     # sequence = 2*WINDOW + 3 = 803 <= 1024
else:
    WINDOW, STRIDE = 800, 400
WINDOW = WINDOW_OVERRIDE or WINDOW
STRIDE = STRIDE_OVERRIDE or STRIDE
SEQ_LEN = 2 * WINDOW + 3 if ATTN_MODE == "echo" else WINDOW + 1
assert SEQ_LEN <= 1024, f"sequence length {SEQ_LEN} exceeds the 1024-token Phase-1 training context"
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | METHOD={METHOD} | ATTN_MODE={ATTN_MODE} | window={WINDOW} stride={STRIDE} seq_len={SEQ_LEN} | "
      f"train={TRAIN_FOLDS} val={VAL_FOLDS} | smoke={SMOKE_TEST}")

In [ ]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

# NER text must reach the tokenizer UNCHANGED (no clean_text / normalisation), otherwise character offsets drift.
_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- the same guard the Phase-1
# Decoder-03 notebook uses. Pre-tokens longer than 1000 bytes are BPE-encoded in 1000-byte pieces. Normal text never has such
# chunks, so its tokenisation (and every character offset) is unchanged; byte-level BPE decodes the pieces to the exact same bytes.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

## Phase-1 model code — copied **verbatim** from `caps_decoder_only_architecture_03.ipynb` (Cells 4-11)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [ ]:
# Cell 4 — Model configuration
#
# One config drives three architectures (all registered, all testable):
#   preset_alibi_mla()  -> the NEW architecture (ALiBi positions + Multi-Head Latent Attention)
#   preset_alibi_mha()  -> ablation: ALiBi with ordinary MHA (isolates the ALiBi effect)
#   preset_rope_mha()   -> corrected baseline: RoPE + MHA (model 01 accidentally had NO positions)

@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 13
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                # only used by mixer_type="attention" (GQA if < n_head)
    head_dim: int = 64
    ff_mult: float = 8/3              # SwiGLU hidden = round64(ff_mult * d_model) = 1408
    d_ff: Optional[int] = None
    max_seq_len: int = 1024           # TRAINING context. ALiBi itself has no length limit.

    # --- components ---
    norm_type: str = "rmsnorm"        # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-5
    pos_type: str = "alibi"           # "alibi" | "rope" | "learned" | "none"
    rope_theta: float = 10000.0
    mixer_type: str = "mla"           # "attention" (MHA/GQA) | "mla"
    kv_latent_dim: int = 128          # d_c: MLA caches only this many numbers per token per layer
    mla_latent_norm: bool = True      # RMSNorm on the latent c_t (as in DeepSeek-V2)
    mlp_type: str = "swiglu"          # "swiglu" | "gelu"
    channel_mixer_type: str = "mlp"

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, "d_model must equal n_head*head_dim"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert self.pos_type in ("alibi", "rope", "learned", "none")
        assert self.mixer_type in ("attention", "mla")
        if self.mixer_type == "mla":
            # RoPE rotates q and k by a position-dependent matrix that sits BETWEEN W_Q and
            # W_UK, which breaks MLA's weight absorption (DeepSeek needs "decoupled RoPE").
            # ALiBi only adds a bias outside q.k, so it composes cleanly with MLA.
            assert self.pos_type != "rope", "MLA + RoPE needs decoupled RoPE; use ALiBi"
            assert not self.bias, "MLA absorption assumes bias-free up-projections"
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


def preset_alibi_mla(vocab_size=None, max_seq_len=1024):
    # MLA saves ~0.33M attention params per layer vs MHA (d_c=128). 13 layers instead of 12
    # reinvests that saving so the model stays parameter-matched to the 12-layer baseline.
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=13, pos_type="alibi",
                       mixer_type="mla", kv_latent_dim=128, max_seq_len=max_seq_len)

def preset_alibi_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=12, pos_type="alibi",
                       mixer_type="attention", max_seq_len=max_seq_len)

def preset_rope_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=12, pos_type="rope",
                       mixer_type="attention", max_seq_len=max_seq_len)

PRESETS = {"alibi_mla": preset_alibi_mla, "alibi_mha": preset_alibi_mha, "rope_mha": preset_rope_mha}
print(json.dumps(asdict(preset_alibi_mla()), indent=2))

In [ ]:
# Cell 5 — Normalisation layers (hand-written)

class LayerNorm(nn.Module):
    """Classic LayerNorm: centre by the mean, scale by the std, learn gain g and bias b."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.b = nn.Parameter(torch.zeros(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return (((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype)) * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: no mean-centring and no bias; x / rms(x) * g. Statistics computed in fp32."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g


def build_norm(cfg: ModelConfig, d: Optional[int] = None):
    d = d or cfg.d_model
    if cfg.norm_type == "layernorm":
        return LayerNorm(d, cfg.norm_eps)
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(d, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [ ]:
# Cell 6 — Positional mechanisms: ALiBi (new) and RoPE (fixed wiring)
#
# ALiBi (Press et al., 2022). No position embeddings at all. Each head h adds a fixed bias
#       s_ij = q_i.k_j / sqrt(d)  -  m_h * (i - j),        j <= i
# Inside the softmax this multiplies attention by exp(-m_h (i-j)): an exponential recency prior.
# Only the distance (i - j) matters: softmax ignores a per-row constant, and
# -m(i-j) = -m*i + m*j, where -m*i is constant along row i.
# Slopes are geometric, m_h = 2^(-8h/H), h = 1..H. For H = 8: 1/2, 1/4, ..., 1/256, so heads
# range from "looks only a few tokens back" to "looks hundreds of tokens back".

def alibi_slopes(n_head: int) -> List[float]:
    def pow2_slopes(n):
        start = 2.0 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_head).is_integer():
        return pow2_slopes(n_head)
    # Non-power-of-2 head counts: the paper's interpolation rule.
    c = 2 ** math.floor(math.log2(n_head))
    return pow2_slopes(c) + pow2_slopes(2 * c)[0::2][: n_head - c]


class ALiBi(nn.Module):
    def __init__(self, n_head: int):
        super().__init__()
        self.register_buffer("slopes", torch.tensor(alibi_slopes(n_head), dtype=torch.float32),
                             persistent=False)

    def bias(self, q_len: int, k_len: int, device) -> torch.Tensor:
        """[1, H, q_len, k_len] float32. Queries are the LAST q_len of the k_len positions
        (so it also works for KV-cached decoding). Includes the causal mask (-inf for j > i)."""
        q_pos = torch.arange(k_len - q_len, k_len, device=device)
        k_pos = torch.arange(k_len, device=device)
        dist = (q_pos[:, None] - k_pos[None, :]).float()          # i - j
        b = -self.slopes.to(device)[:, None, None] * dist          # [H, Tq, Tk]
        b = b.masked_fill(dist < 0, float("-inf"))
        return b.unsqueeze(0)


class RoPE(nn.Module):
    """Rotary embeddings. Table is built for 2x the training length so generation never overflows."""
    def __init__(self, head_dim: int, max_pos: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_pos).float(), inv_freq)   # [P, D/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def tables(self, position_ids: torch.Tensor):
        """position_ids [B, T] -> (cos, sin) each [B, 1, T, D/2]."""
        return self.cos[position_ids].unsqueeze(1), self.sin[position_ids].unsqueeze(1)

    @staticmethod
    def apply(x, cos, sin):
        """x [B, H, T, D]; rotation done in fp32 for stability."""
        dt = x.dtype
        x = x.float()
        x1, x2 = x[..., 0::2], x[..., 1::2]
        out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)
        return out.to(dt)

In [ ]:
# Cell 7 — Shared attention kernel + standard MHA/GQA token mixer
#
# The Backbone builds ONE additive mask per forward pass (`attn_bias`, shape [B or 1, H or 1, T, S])
# containing ALiBi + causal + padding, and passes it to every layer. Every token mixer then
# calls `attend`. When attn_bias is None we use SDPA's fast is_causal path.

def attend(q, k, v, attn_bias, is_causal, scale, dropout_p=0.0, use_sdpa=True):
    """q [B,H,T,Dq], k [B,H,S,Dq], v [B,H,S,Dv] -> [B,H,T,Dv]."""
    if attn_bias is not None:
        attn_bias = attn_bias.to(q.dtype)          # SDPA requires mask dtype == query dtype
        is_causal = False
    if use_sdpa:
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                                              dropout_p=dropout_p, is_causal=is_causal,
                                              scale=scale)
    # Manual reference path (used by the tests to cross-check SDPA).
    scores = (q.float() @ k.float().transpose(-2, -1)) * scale
    if is_causal:
        T, S = q.shape[-2], k.shape[-2]
        scores = scores.masked_fill(
            torch.triu(torch.ones(T, S, dtype=torch.bool, device=q.device), diagonal=S - T + 1),
            float("-inf"))
    if attn_bias is not None:
        scores = scores + attn_bias.float()
    p = F.softmax(scores, dim=-1)
    if dropout_p > 0:
        p = F.dropout(p, dropout_p)
    return (p @ v.float()).to(q.dtype)


class Attention(nn.Module):
    """Multi-head (n_kv_head == n_head) or grouped-query (n_kv_head < n_head) self-attention.
    Cache format: (k, v), each [B, n_kv_head, S, head_dim]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.head_dim).transpose(1, 2)

    @staticmethod
    def cache_len(cache):
        return cache[0].shape[2]

    def cache_numbers_per_token(self):
        return 2 * self.n_kv_head * self.head_dim

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(x), self.n_kv_head)
        v = self._split(self.v_proj(x), self.n_kv_head)
        if rope_cs is not None:                       # FIX: RoPE is now actually applied
            cos, sin = rope_cs
            q, k = RoPE.apply(q, cos, sin), RoPE.apply(k, cos, sin)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        out = attend(q, k, v, attn_bias, is_causal=(attn_bias is None and T > 1),
                     scale=self.scale,
                     dropout_p=self.cfg.attn_dropout if self.training else 0.0,
                     use_sdpa=self.cfg.use_sdpa)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [ ]:
# Cell 8 — Multi-Head Latent Attention (MLA, DeepSeek-V2), adapted to ALiBi
#
#   c_t      = Norm(W_DKV x_t)          in R^{d_c}        (the ONLY thing we cache)
#   k_t^(h)  = W_UK^(h) c_t,   v_t^(h) = W_UV^(h) c_t    (every head gets its own K, V)
#   q_t^(h)  = W_Q^(h) x_t
#
# KV cache per token per layer: MHA stores 2*H*D_h = 1024 numbers; MLA stores d_c = 128 (8x less).
#
# MLA vs GQA: GQA *ties* heads (groups share identical K,V). MLA gives each head distinct K,V,
# all reconstructed from a shared rank-d_c subspace. With d_c = 2*n_kv*D_h, MLA can exactly
# express GQA (the up-projections just copy slices), so at equal cache size GQA is a special case.
#
# Weight absorption (inference trick, implemented below as `absorb=True`):
#   q_i^(h).k_j^(h) = (W_Q^(h) x_i)^T W_UK^(h) c_j = [W_UK^(h)T q_i^(h)] . c_j
# so we can attend directly against the cached latents without ever building K:
#   q_lat = W_UK^T q,  scores = q_lat . c,  out_lat = softmax(scores) c,  out^(h) = W_UV^(h) out_lat
# RoPE would insert a position-dependent rotation R_{i-j} between W_Q and W_UK and break this;
# ALiBi's bias lives outside the dot product, so absorption is unaffected.
# Training uses the materialised path (simpler, identical maths); a test checks both agree.

class MLAttention(nn.Module):
    """Cache format: c, [B, S, d_c]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        H, Dh, dc = cfg.n_head, cfg.head_dim, cfg.kv_latent_dim
        self.n_head, self.head_dim, self.d_c = H, Dh, dc
        self.scale = 1.0 / math.sqrt(Dh)              # scores are still per-head q.k with dim D_h
        self.q_proj = nn.Linear(cfg.d_model, H * Dh, bias=False)
        self.kv_down = nn.Linear(cfg.d_model, dc, bias=False)          # W_DKV
        self.kv_norm = build_norm(cfg, dc) if cfg.mla_latent_norm else nn.Identity()
        self.k_up = nn.Linear(dc, H * Dh, bias=False)                  # W_UK (all heads)
        self.v_up = nn.Linear(dc, H * Dh, bias=False)                  # W_UV (all heads)
        self.o_proj = nn.Linear(H * Dh, cfg.d_model, bias=False)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)
        self.absorb = False                           # runtime switch (inference only)

    @staticmethod
    def cache_len(cache):
        return cache.shape[1]

    def cache_numbers_per_token(self):
        return self.d_c

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        assert rope_cs is None, "MLA is used with ALiBi (or no positions), not RoPE"
        B, T, _ = x.shape
        H, Dh = self.n_head, self.head_dim
        c = self.kv_norm(self.kv_down(x))                         # [B, T, d_c]
        if kv_cache is not None:
            c = torch.cat([kv_cache, c], dim=1)                   # [B, S, d_c]
        new_cache = c
        S = c.shape[1]
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)      # [B, H, T, Dh]
        is_causal = (attn_bias is None and T > 1)
        dp = self.cfg.attn_dropout if self.training else 0.0

        if not self.absorb:
            k = self.k_up(c).view(B, S, H, Dh).transpose(1, 2)    # [B, H, S, Dh]
            v = self.v_up(c).view(B, S, H, Dh).transpose(1, 2)
            out = attend(q, k, v, attn_bias, is_causal, self.scale, dp, self.cfg.use_sdpa)
        else:
            W_uk = self.k_up.weight.view(H, Dh, self.d_c)         # rows of head h -> W_UK^(h)
            W_uv = self.v_up.weight.view(H, Dh, self.d_c)
            q_lat = torch.einsum("bhtd,hdc->bhtc", q, W_uk.to(q.dtype))   # W_UK^T q
            c_h = c.unsqueeze(1).expand(B, H, S, self.d_c)
            out_lat = attend(q_lat, c_h, c_h, attn_bias, is_causal, self.scale, dp,
                             self.cfg.use_sdpa)                   # [B, H, T, d_c]
            out = torch.einsum("bhtc,hdc->bhtd", out_lat, W_uv.to(out_lat.dtype))

        out = out.transpose(1, 2).contiguous().view(B, T, H * Dh)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [ ]:
# Cell 9 — Channel mixers, registries, transformer block

class SwiGLU(nn.Module):
    """down( silu(gate(x)) * up(x) )."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))


TOKEN_MIXERS: Dict[str, Any] = {}
CHANNEL_MIXERS: Dict[str, Any] = {}

def register_token_mixer(name):
    def deco(fn):
        TOKEN_MIXERS[name] = fn
        return fn
    return deco

def register_channel_mixer(name):
    def deco(fn):
        CHANNEL_MIXERS[name] = fn
        return fn
    return deco

# FIX: token mixers no longer receive `rope` at build time (model 01 silently dropped it there).
# Positional information now flows through forward(): rope_cs / attn_bias, built in the Backbone.
@register_token_mixer("attention")
def _build_attention(cfg):
    return Attention(cfg)

@register_token_mixer("mla")
def _build_mla(cfg):
    return MLAttention(cfg)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    return {"swiglu": SwiGLU, "gelu": GELUMLP}[cfg.mlp_type](cfg)


class Block(nn.Module):
    """Pre-norm block: x = x + mixer(norm(x)); x = x + mlp(norm(x))."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = build_norm(cfg)
        self.mixer = TOKEN_MIXERS[cfg.mixer_type](cfg)
        self.norm2 = build_norm(cfg)
        self.cmix = CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)

    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [ ]:
# Cell 10 — Backbone and language model

class Backbone(nn.Module):
    """Embedding -> N blocks -> final norm. Builds positional info and masks ONCE per forward."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.alibi = ALiBi(cfg.n_head) if cfg.pos_type == "alibi" else None
        self.rope = RoPE(cfg.head_dim, 2 * cfg.max_seq_len, cfg.rope_theta) \
            if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(2 * cfg.max_seq_len, cfg.d_model) \
            if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def _attn_bias(self, B, T, past, attention_mask, device):
        """Additive mask [B|1, H|1, T, S] or None (=> SDPA fast causal path)."""
        S = past + T
        need = (self.alibi is not None) or (attention_mask is not None) or (past > 0 and T > 1)
        if not need:
            return None
        if self.alibi is not None:
            bias = self.alibi.bias(T, S, device)                          # ALiBi + causal
        else:
            q_pos = torch.arange(past, S, device=device)
            k_pos = torch.arange(S, device=device)
            bias = torch.zeros(T, S, device=device).masked_fill(
                k_pos[None, :] > q_pos[:, None], float("-inf"))[None, None]
        if attention_mask is not None:
            # Mask padded KEYS. A padded QUERY would then have every key masked -> softmax of
            # all -inf -> NaN, which can leak into later layers through 0*NaN. So let each
            # padded query attend to itself (its output is never used).
            key_pad = ~attention_mask.bool()                                   # [B, S]
            q_idx = torch.arange(past, S, device=device)
            k_idx = torch.arange(S, device=device)
            is_self = q_idx[:, None] == k_idx[None, :]                         # [T, S]
            block = key_pad[:, None, :] & ~is_self[None]                       # [B, T, S]
            pad = torch.zeros(B, 1, T, S, device=device).masked_fill(block[:, None], float("-inf"))
            bias = bias + pad
        return bias

    def forward(self, input_ids, attention_mask=None, kv_caches=None):
        B, T = input_ids.shape
        dev = input_ids.device
        past = 0 if kv_caches is None else self.blocks[0].mixer.cache_len(kv_caches[0])
        x = self.tok_emb(input_ids)

        rope_cs = None
        if self.rope is not None or self.pos_emb is not None:
            if attention_mask is not None:       # left padding: real tokens start at position 0
                pos = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)[:, -T:]
            else:
                pos = torch.arange(past, past + T, device=dev).unsqueeze(0).expand(B, T)
            if self.rope is not None:
                rope_cs = self.rope.tables(pos)
            else:
                x = x + self.pos_emb(pos)

        attn_bias = self._attn_bias(B, T, past, attention_mask, dev)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, attn_bias=attn_bias, rope_cs=rope_cs,
                        kv_cache=None if kv_caches is None else kv_caches[i])
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def set_mla_absorb(self, flag: bool):
        for blk in self.backbone.blocks:
            if isinstance(blk.mixer, MLAttention):
                blk.mixer.absorb = flag

    def forward(self, input_ids, targets=None, attention_mask=None, kv_caches=None,
                last_only=False, reduction="mean"):
        h, new_caches = self.backbone(input_ids, attention_mask=attention_mask,
                                      kv_caches=kv_caches)
        if last_only:                 # generation: only the last position's logits are needed
            h = h[:, -1:, :]
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1),
                                   ignore_index=-100, reduction=reduction)
            if reduction == "none":
                loss = loss.view(targets.shape)
        return logits, loss, new_caches

In [ ]:
# Cell 11 — Init, parameter counting, optimiser groups, checkpoint I/O

def init_weights(model: LegalLM, cfg: ModelConfig):
    """N(0, 0.02) everywhere; residual-branch output projections scaled by 1/sqrt(2L)
    so the residual stream's variance does not grow with depth."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    with torch.no_grad():
        for blk in model.backbone.blocks:
            blk.mixer.o_proj.weight.mul_(scale)
            blk.cmix.down_proj.weight.mul_(scale)


def count_params(model: LegalLM) -> Dict[str, int]:
    seen, total = set(), 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p))
            total += p.numel()
    emb = model.backbone.tok_emb.weight.numel()
    attn = sum(p.numel() for blk in model.backbone.blocks for p in blk.mixer.parameters())
    return {"total": total, "embedding": emb, "non_embedding": total - emb, "attention": attn}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Decay only 2-D weight matrices (not embeddings, norm gains or biases)."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "tok_emb" not in name and "pos_emb" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def save_model(path, model: LegalLM, cfg: ModelConfig, extra: dict = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "extra": extra or {}}, path)


def load_model(path, map_location="cpu") -> Tuple[LegalLM, ModelConfig, dict]:
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})

## Load the Phase-1 checkpoint

In [ ]:
# Cell 12 — Load the Phase-1 checkpoint and sanity-check it
_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
lm_cfg = ModelConfig(**_ck["config"])
lm = LegalLM(lm_cfg)
lm.load_state_dict(_ck["model"])
lm_extra = _ck.get("extra", {})
del _ck
lm = lm.to(device).eval()
pc = count_params(lm)
print(f"Phase-1 model | pos_type={lm_cfg.pos_type} mixer={lm_cfg.mixer_type} (latent {lm_cfg.kv_latent_dim}) | "
      f"layers={lm_cfg.n_layer} d_model={lm_cfg.d_model} heads={lm_cfg.n_head} d_ff={lm_cfg.d_ff} | "
      f"params {pc['total']/1e6:.2f}M (embedding {pc['embedding']/1e6:.2f}M)")
print("checkpoint extra: step", lm_extra.get("step"), "| val_loss", lm_extra.get("val_loss"))

# If the weights did not load properly, the LM loss on a normal legal sentence would sit near ln(V) ~ 10.4.
_sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
_ids = [BOS_ID] + tokenizer.encode(_sent)
_x = torch.tensor([_ids[:-1]], device=device); _y = torch.tensor([_ids[1:]], device=device)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    _, _l, _ = lm(_x, targets=_y)
print(f"LM loss on a legal sentence: {_l.item():.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
assert _l.item() < 9.0, "checkpoint looks untrained / mis-loaded"

if ATTN_MODE in ("bidir", "bidir_top") and lm_cfg.pos_type != "alibi":
    raise RuntimeError(f"ATTN_MODE='{ATTN_MODE}' is implemented for ALiBi checkpoints (pos_type='alibi'); this one has "
                       f"pos_type='{lm_cfg.pos_type}'. Use ATTN_MODE='echo' or 'causal'.")
if METHOD == "partial":
    assert 0 <= PARTIAL_K <= lm_cfg.n_layer, "PARTIAL_K must be between 0 and the number of layers"

## Data, alignment and metrics
**Character offsets are everything here:** a span that is off by one character scores zero. Cell 14 contains the oracle test that proves the token↔character mapping is lossless.

In [ ]:
# Cell 13 — Phase-2 data: labels, folds, gold parsing, offset-convention check
from huggingface_hub import hf_hub_download

P2 = "Phase_2_LNER"
def fetch(rel: str) -> Path:
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet setting, or download the Phase_2_LNER folder and set LOCAL_DATA_DIR.") from ex

def load_json(rel):
    return json.loads(fetch(rel).read_text(encoding="utf-8"))

SPEC_LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
labels = [ln.split()[0] for ln in fetch(f"{P2}/Train/labels.txt").read_text(encoding="utf-8").splitlines() if ln.strip()]
if set(labels) != set(SPEC_LABELS):
    print("WARNING: labels.txt differs from the 12 labels in the capstone document:", labels)
LABEL_SET = set(labels)
print(len(labels), "labels:", labels)

def _text_of(v):
    return v if isinstance(v, str) else v.get("text", "")

def _parse_one(e):
    """One gold entity -> (start, end, label, text|None). Accepts [s,e,label(,text)] or dict forms."""
    if isinstance(e, dict):
        low = {k.lower(): v for k, v in e.items()}
        s  = next((low[k] for k in ("start", "start_char", "start_offset", "begin", "s") if k in low), None)
        en = next((low[k] for k in ("end", "end_char", "end_offset", "stop", "e") if k in low), None)
        lab = next((low[k] for k in ("label", "type", "tag", "entity", "category") if isinstance(low.get(k), str) and low[k] in LABEL_SET), None)
        tx = next((low[k] for k in ("text", "entity_text", "span", "mention", "value") if isinstance(low.get(k), str)), None)
    else:
        ints = [x for x in e if isinstance(x, int) and not isinstance(x, bool)]
        strs = [x for x in e if isinstance(x, str)]
        s, en = (ints[0], ints[1]) if len(ints) >= 2 else (None, None)
        lab = next((x for x in strs if x in LABEL_SET), None)
        tx = next((x for x in strs if x not in LABEL_SET), None)
    if s is None or en is None or lab is None:
        raise ValueError(f"cannot parse gold entity {e!r} -- paste this line to Claude and adapt _parse_one")
    return int(s), int(en), lab, tx

def parse_gold(raw):
    if isinstance(raw, list):                                   # list of records
        raw = {str(r.get("id", r.get("doc_id"))): r.get("entities", r.get("annotations", r.get("spans", []))) for r in raw}
    out = {}
    for did, ents in raw.items():
        if isinstance(ents, dict):
            ents = ents.get("entities", ents.get("annotations", []))
        out[str(did)] = [_parse_one(e) for e in ents]
    return out

TEXTS, GOLD_RAW = {}, {}
for f in (1, 2):
    TEXTS[f] = {str(k): _text_of(v) for k, v in load_json(f"{P2}/Train/fold{f}-text.json").items()}
    GOLD_RAW[f] = parse_gold(load_json(f"{P2}/Train/fold{f}-gold.json"))
TEST_TEXTS = {str(k): _text_of(v) for k, v in load_json(f"{P2}/Test/fold3-text.json").items()}
print({f: len(TEXTS[f]) for f in TEXTS}, "| test docs:", len(TEST_TEXTS))

# --- sanity: is this really the PHASE-2 test set? (Phase-1 test ids are the strings "1".."1000"; Phase-2 ids are long numbers like "1128835") ---
_tids = list(TEST_TEXTS)
print(f"PHASE-2 TEST SET: {len(_tids)} documents | first ids: {_tids[:3]} | train sample ids: {list(TEXTS[1])[:2]}")
if set(_tids) & (set(TEXTS[1]) | set(TEXTS[2])):
    raise RuntimeError("some test document ids also occur in the training folds: wrong test file?")
if len(_tids) >= 100 and all(i.isdigit() and int(i) <= 5000 for i in _tids):
    raise RuntimeError("these test ids look like the PHASE-1 prompt ids (1..1000), not Phase-2 document ids. "
                       "Check HF_REPO / LOCAL_DATA_DIR: the file must be Phase_2_LNER/Test/fold3-text.json.")
_first = next(iter(GOLD_RAW[1].items()))
print("sample gold:", _first[0], _first[1][:3])

# --- is the gold `end` exclusive (text[s:e]) or inclusive (text[s:e+1])? ---------------------------------------
ex = inc = tot = 0
for f in (1, 2):
    for did, ents in GOLD_RAW[f].items():
        t = TEXTS[f].get(did)
        for s, e, l, tx in ents:
            if t is None or tx is None: continue
            tot += 1; ex += (t[s:e] == tx); inc += (t[s:e + 1] == tx)
print(f"end-offset convention check over {tot} entities: exclusive matches {ex}, inclusive matches {inc}")
END_INCLUSIVE = inc > ex
if tot and max(ex, inc) / tot < 0.9:
    print("WARNING: <90% of gold entities reproduce their text from the offsets. Check for text normalisation / offset base.")
print("END_INCLUSIVE =", END_INCLUSIVE, "(predictions are converted back to this convention at submission time)")

GOLD = {}                                                        # fold -> {doc_id: [(s, e_exclusive, label)]}
bad = 0
for f in (1, 2):
    GOLD[f] = {}
    for did, ents in GOLD_RAW[f].items():
        if did not in TEXTS[f]:
            bad += 1; continue
        L = len(TEXTS[f][did]); keep = []
        for s, e, l, tx in ents:
            e = e + 1 if END_INCLUSIVE else e
            if 0 <= s < e <= L: keep.append((s, e, l))
            else: bad += 1
        GOLD[f][did] = keep
    for did in TEXTS[f]:
        GOLD[f].setdefault(did, [])
print("entities dropped for bad offsets / unknown doc:", bad)
GOLD_ALL = {did: g for f in (1, 2) for did, g in GOLD[f].items()}
ALL_TEXTS = {did: t for f in (1, 2) for did, t in TEXTS[f].items()}
assert len(GOLD_ALL) == len(GOLD[1]) + len(GOLD[2]), "document ids collide between fold 1 and fold 2"
cnt = Counter(l for f in TRAIN_FOLDS for ents in GOLD[f].values() for _, _, l in ents)
print("train-fold entity counts:", dict(cnt.most_common()))

In [ ]:
# Cell 14 — Tokenise with character offsets, BIO tags, BIO decoding, and the ORACLE ceiling test
TAGS = ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
TAG2ID = {t: i for i, t in enumerate(TAGS)}
NT = len(TAGS)
TAG_PREFIX = [t.split("-", 1)[0] if t != "O" else "O" for t in TAGS]
TAG_LABEL  = [t.split("-", 1)[1] if t != "O" else None for t in TAGS]

# The BPE pre-tokeniser groups punctuation into runs (e.g. ".," or ")."), so a token can overshoot an entity edge by one
# punctuation character. We learn from the TRAIN gold which edge characters NEVER start / end an entity and trim them.
TRIM_END, TRIM_START = frozenset(), frozenset()
_END_CANDIDATES, _START_CANDIDATES = ",;:!?)]}\"'\u201d\u2019-\u2013\u2014/", "([{\"'\u201c\u2018,;:.-\u2013\u2014/"

def learn_trim_chars(gold, texts):
    ends, starts = Counter(), Counter()
    for did, ents in gold.items():
        t = texts[did]
        for s, e, l in ents:
            ends[t[e - 1]] += 1; starts[t[s]] += 1
    te = frozenset(c for c in _END_CANDIDATES if ends[c] == 0)
    ts = frozenset(c for c in _START_CANDIDATES if starts[c] == 0)
    return te, ts

def tokenise(text: str):
    ids, offs = tokenizer.encode(text, return_offsets=True)        # offsets are CHARACTER offsets into `text`
    return np.asarray(ids, dtype=np.int32), np.asarray(offs, dtype=np.int64).reshape(-1, 2)

def spans_to_tags(offs, ents):
    """Gold char spans -> one tag per token. A token belongs to an entity if their char ranges overlap.
    Returns (tags, number_of_entities_dropped_because_they_overlap_an_earlier_entity)."""
    N = len(offs); tags = np.zeros(N, dtype=np.int64); dropped = 0
    if N == 0: return tags, len(ents)
    ts, te = offs[:, 0], offs[:, 1]
    for s, e, l in sorted(ents, key=lambda x: (x[0], -(x[1] - x[0]))):
        lo = int(np.searchsorted(te, s, side="right"))             # first token with end > s
        hi = int(np.searchsorted(ts, e, side="left"))              # first token with start >= e
        if lo >= hi or (tags[lo:hi] != 0).any():
            dropped += 1; continue
        tags[lo] = TAG2ID["B-" + l]; tags[lo + 1:hi] = TAG2ID["I-" + l]
    return tags, dropped

def tags_to_spans(text, offs, tag_ids):
    """Token tags -> character spans (start, end_exclusive, label). Lenient: I- after O/other label opens a span.
    Leading/trailing whitespace is trimmed because the tokenizer attaches the space to the FOLLOWING word;
    TRIM_START / TRIM_END remove punctuation that a fused punctuation-run token pulled into the span."""
    spans, cur = [], None
    for i, t in enumerate(tag_ids):
        t = int(t)
        if t == 0:
            if cur is not None: spans.append(cur); cur = None
            continue
        l = TAG_LABEL[t]
        if TAG_PREFIX[t] == "B" or cur is None or cur[0] != l:
            if cur is not None: spans.append(cur)
            cur = [l, i, i]
        else:
            cur[2] = i
    if cur is not None: spans.append(cur)
    out = []
    for l, a, b in spans:
        s, e = int(offs[a, 0]), int(offs[b, 1])
        while True:                                                 # trim whitespace + learned edge punctuation until stable
            s0, e0 = s, e
            while s < e and (text[s].isspace() or text[s] in TRIM_START): s += 1
            while e > s and (text[e - 1].isspace() or text[e - 1] in TRIM_END): e -= 1
            if (s, e) == (s0, e0): break
        if e > s: out.append((s, e, l))
    return out

def docs_from_texts(texts, gold=None):
    docs, dropped, t0 = [], 0, time.time()
    for did, text in texts.items():
        ids, offs = tokenise(text)
        d = dict(id=did, text=text, ids=ids, offs=offs, tags=None)
        if gold is not None:
            d["tags"], dr = spans_to_tags(offs, gold[did]); dropped += dr
        docs.append(d)
    print(f"tokenised {len(docs)} docs in {time.time()-t0:.0f}s | {sum(len(d['ids']) for d in docs)/1e6:.2f}M tokens"
          + (f" | entities dropped (nested/overlapping): {dropped}" if gold is not None else ""))
    return docs

def build_docs(folds, cache_name=None):
    texts = {did: t for f in folds for did, t in TEXTS[f].items()}
    assert len(texts) == sum(len(TEXTS[f]) for f in folds), "document ids collide between folds"
    key = (cache_name, len(texts), sum(len(t) for t in texts.values()))
    cp = WORK_DIR / f"tokcache_{cache_name}.pkl"
    if cache_name and cp.exists():
        c = pickle.load(open(cp, "rb"))
        if c["key"] == key: return c["docs"]
    docs = docs_from_texts(texts, GOLD_ALL)
    if cache_name: pickle.dump(dict(key=key, docs=docs), open(cp, "wb"))
    return docs

In [ ]:
# Cell 15 — Strict span-level metrics (exact start, end and label) + error breakdown
def strict_scores(gold, pred):
    """gold/pred: {doc_id: iterable of (start, end, label)}. Macro-F1 is averaged over labels that occur in the gold."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for did in set(gold) | set(pred):
        g, p = set(gold.get(did, ())), set(pred.get(did, ()))
        for x in g & p: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    per = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = dict(p=P, r=R, f1=(2 * P * R / (P + R)) if P + R else 0.0, tp=tp[l], fp=fp[l], fn=fn[l], support=tp[l] + fn[l])
    present = [l for l in labels if per[l]["support"] > 0]
    TP, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    mP = TP / (TP + FP) if TP + FP else 0.0; mR = TP / (TP + FN) if TP + FN else 0.0
    return dict(per=per,
                macro_f1=float(np.mean([per[l]["f1"] for l in present])) if present else 0.0,
                macro_p=float(np.mean([per[l]["p"] for l in present])) if present else 0.0,
                macro_r=float(np.mean([per[l]["r"] for l in present])) if present else 0.0,
                macro_f1_all12=float(np.mean([per[l]["f1"] for l in labels])),
                micro_f1=(2 * mP * mR / (mP + mR)) if mP + mR else 0.0)

def error_breakdown(gold, pred):
    """Why does each missed gold span count as wrong?  Answers 'boundary error or label error?'."""
    c = Counter()
    for did in gold:
        g = set(gold[did]); p = set(pred.get(did, ()))
        plist = sorted(p)
        for (s, e, l) in g - p:
            same_span = [x for x in plist if x[0] == s and x[1] == e]
            over = [x for x in plist if x[0] < e and s < x[1]]
            if same_span:                      c["wrong_label_same_span"] += 1
            elif any(x[2] == l for x in over): c["boundary_error_same_label"] += 1
            elif over:                         c["boundary_and_label_error"] += 1
            else:                              c["missed_entirely"] += 1
    return dict(c)

def print_scores(sc, title=""):
    print(f"{title} strict macro-F1 {sc['macro_f1']:.4f} | P {sc['macro_p']:.4f} R {sc['macro_r']:.4f} | "
          f"micro-F1 {sc['micro_f1']:.4f} | macro-F1 over all 12 labels {sc['macro_f1_all12']:.4f}")
    print(f"  {'label':10s} {'supp':>6s} {'P':>6s} {'R':>6s} {'F1':>6s}")
    for l in labels:
        d = sc["per"][l]
        print(f"  {l:10s} {d['support']:6d} {d['p']:6.3f} {d['r']:6.3f} {d['f1']:6.3f}")

# ---- ORACLE test: gold -> tags -> spans must reproduce the gold. This bounds what ANY model can score. ----------------
def oracle_test(docs, gold):
    global TRIM_END, TRIM_START
    if TRIM_PUNCT:
        _te, _ts = TRIM_END, TRIM_START
        TRIM_END, TRIM_START = frozenset(), frozenset()
        base = strict_scores({d["id"]: gold[d["id"]] for d in docs},
                             {d["id"]: tags_to_spans(d["text"], d["offs"], d["tags"]) for d in docs})
        print(f"ORACLE without punctuation trimming: macro-F1 = {base['macro_f1']:.4f}")
        TRIM_END, TRIM_START = _te, _ts
    pred = {d["id"]: tags_to_spans(d["text"], d["offs"], d["tags"]) for d in docs}
    g = {d["id"]: gold[d["id"]] for d in docs}
    sc = strict_scores(g, pred)
    print(f"ORACLE (tokenisation ceiling): strict macro-F1 = {sc['macro_f1']:.4f} | micro-F1 = {sc['micro_f1']:.4f}")
    bad = [(l, sc["per"][l]["f1"], sc["per"][l]["fn"]) for l in labels if sc["per"][l]["support"] and sc["per"][l]["f1"] < 0.99]
    if bad: print("  labels below 0.99 (token boundaries do not line up with the entity edges):", bad)
    shown = 0
    for d in docs:
        miss = set(g[d["id"]]) - set(pred[d["id"]])
        for (s, e, l) in sorted(miss)[:2]:
            if shown < 5:
                print(f"  lost: {l} gold={d['text'][s:e]!r} -> tokens cover {d['text'][max(0,s-3):e+3]!r}"); shown += 1
    return sc

In [ ]:
# Cell 16 — Windows and sequence layout (pure python / numpy)
def make_windows(n_tok, W, S):
    """Overlapping windows (start, end) that cover every token."""
    if n_tok == 0: return []
    if n_tok <= W: return [(0, n_tok)]
    starts = list(range(0, n_tok - W + 1, S))
    if starts[-1] + W < n_tok: starts.append(n_tok - W)
    return [(a, a + W) for a in starts]

def build_seq(win_ids, win_tags=None):
    """Returns (input_ids, offset, labels|None). Logits at positions [offset : offset+len(win_ids)] belong to the window tokens.
       causal/bidir : [BOS] w
       echo         : [BOS] w [EOS] [BOS] w      (tags are read from the SECOND copy, which has seen the whole window)"""
    n = len(win_ids)
    if ATTN_MODE == "echo":
        seq = [BOS_ID] + list(win_ids) + [EOS_ID, BOS_ID] + list(win_ids)
        off = n + 3
    else:
        seq = [BOS_ID] + list(win_ids)
        off = 1
    lab = None
    if win_tags is not None:
        lab = [-100] * off + [int(t) for t in win_tags]
    return seq, off, lab

class WindowMerger:
    """Averages the per-token probabilities of every window that covers a token."""
    def __init__(self, docs):
        self.acc = [np.zeros((len(d["ids"]), NT), dtype=np.float32) for d in docs]
        self.cnt = [np.zeros(len(d["ids"]), dtype=np.float32) for d in docs]
    def add(self, di, a, b, p):
        self.acc[di][a:b] += p; self.cnt[di][a:b] += 1
    def result(self):
        return [self.acc[i] / np.maximum(self.cnt[i], 1)[:, None] for i in range(len(self.acc))]

def pad_batch(seqs, labs=None):
    L = max(len(s) for s in seqs)
    ids = np.full((len(seqs), L), PAD_ID, dtype=np.int64); mask = np.zeros((len(seqs), L), dtype=np.int64)
    lab = np.full((len(seqs), L), -100, dtype=np.int64) if labs is not None else None
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = s; mask[i, :len(s)] = 1
        if labs is not None: lab[i, :len(labs[i])] = labs[i]
    return ids, mask, lab

## Model, adaptation methods (partial / full / LoRA / LoRMA / norm-only), training

In [ ]:
# Cell 17 — NER model, bidirectional ALiBi (full or top-K blocks only), and the five adaptation methods
import copy

# ---- (a) Attention modes ---------------------------------------------------------------------------------------------------
# Phase-1 Backbone builds ONE additive bias [B,H,T,S] = ALiBi (-m_h (i-j)) + causal (-inf for j>i) + padding, and gives it to
# every block. Here we replace Backbone.forward by a version that can hand DIFFERENT biases to different blocks:
#   blocks < bidir_from : the original causal ALiBi bias           (frozen lower blocks keep their Phase-1 behaviour)
#   blocks >= bidir_from: symmetric ALiBi  -m_h |i-j|, no causal part (right context is visible)
def _bidir_alibi_bias(bb, B, T, attention_mask, dev):
    pos = torch.arange(T, device=dev)
    dist = (pos[:, None] - pos[None, :]).abs().float()                           # |i - j|
    bias = (-bb.alibi.slopes.to(dev)[:, None, None] * dist).unsqueeze(0)         # [1, H, T, T]
    if attention_mask is not None:                                               # mask padded KEYS (real keys always remain, so no NaN rows)
        bias = bias.expand(B, -1, -1, -1).masked_fill(~attention_mask.bool()[:, None, None, :], float("-inf"))
    return bias

def _hybrid_forward(self, input_ids, attention_mask=None, kv_caches=None):
    assert kv_caches is None, "the NER notebook never uses a KV cache"
    B, T = input_ids.shape
    dev = input_ids.device
    x = self.emb_drop(self.tok_emb(input_ids))
    bias_causal = Backbone._attn_bias(self, B, T, 0, attention_mask, dev) if self.bidir_from > 0 else None
    bias_bidir  = _bidir_alibi_bias(self, B, T, attention_mask, dev) if self.bidir_from < len(self.blocks) else None
    for i, blk in enumerate(self.blocks):
        x, _ = blk(x, attn_bias=bias_bidir if i >= self.bidir_from else bias_causal, rope_cs=None, kv_cache=None)
    return self.norm_f(x), []

def set_attention_mode(bb):
    """Install the attention mode on a Backbone instance. causal / echo keep the untouched Phase-1 forward."""
    n = len(bb.blocks)
    if ATTN_MODE == "bidir":
        bb.bidir_from = 0
    elif ATTN_MODE == "bidir_top":
        bb.bidir_from = max(0, n - BIDIR_TOP_K)
    else:
        return
    bb.forward = types.MethodType(_hybrid_forward, bb)
    print(f"attention: blocks 0..{bb.bidir_from - 1} causal | blocks {bb.bidir_from}..{n - 1} bidirectional (symmetric ALiBi)")


# ---- (b) NER model: Phase-1 backbone + Linear head ---------------------------------------------------------------------------
class NERModel(nn.Module):
    """Backbone (embedding -> blocks -> final RMSNorm) + dropout + Linear(d_model -> n_tags): the 'one extra output layer on
    token representations T_i' recipe of Module 2.4."""
    def __init__(self, backbone, d_model, n_tags, dropout):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, n_tags)
        nn.init.normal_(self.head.weight, 0.0, 0.02); nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask):
        h, _ = self.backbone(input_ids, attention_mask=attention_mask)
        return self.head(self.drop(h))                      # [B, T, n_tags]


# ---- (c) LoRA (additive) --------------------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))).  W frozen; B = 0 at start."""
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


# ---- (d) LoRMA (multiplicative; Bihany, Patel & Modi, ACL Findings 2025) -----------------------------------------------------
_PI_IDX = {}
def _pi_index(d, dev):
    """idx[i, j] = (j - i) mod d.  gather(M, 1, idx)[i, j] = M[i, (j - i) mod d]  ==  row i of M rotated right by i."""
    k = (d, str(dev))
    if k not in _PI_IDX:
        i = torch.arange(d, device=dev).view(d, 1); j = torch.arange(d, device=dev).view(1, d)
        _PI_IDX[k] = (j - i) % d
    return _PI_IDX[k]

class LoRMALinear(nn.Module):
    """h = I(B A) (W0 x).  W0 frozen.  A in R^{r x d_out}, B in R^{d_out x r}, d_out = base.out_features.
       'plus' (LoRMA+):  I(BA) = I_d + (alpha/r) BA, computed as y + (alpha/r) B(A y); B = 0 at start.
       'pi'   (LoRMA_pi): row i of BA rotated cyclically by i; B[:,0]=1, A[0,0]=1, rest of A = 0 => I(BA) = I_d at start."""
    def __init__(self, base: nn.Linear, r: int, alpha: float, variant: str):
        super().__init__()
        assert variant in ("plus", "pi"), variant
        d = base.out_features
        assert r < d
        self.base, self.variant, self.d, self.scale = base, variant, d, alpha / r
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); nn.init.zeros_(self.B)
        else:
            nn.init.zeros_(self.A)
            nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
            with torch.no_grad():
                self.A[0, 0] = 1.0
                self.B[:, 0] = 1.0

    def forward(self, x):
        y = self.base(x)                                              # W0 x (frozen); the multiplier acts on the OUTPUT space
        if self.variant == "plus":
            return y + (y @ self.A.t() @ self.B.t()) * self.scale
        M = self.B @ self.A
        Tm = torch.gather(M, 1, _pi_index(self.d, M.device))
        return y @ Tm.t()


def apply_adapter(module, targets, make):
    """Replace every nn.Linear whose attribute name is in `targets` by make(linear). Returns how many were replaced."""
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n


def build_ner_model():
    """Fresh copy of the Phase-1 weights every call (several runs in one session do not contaminate each other)."""
    backbone = copy.deepcopy(lm.backbone).cpu()
    ref = copy.deepcopy(backbone) if METHOD in ("lora", "lorma") else None     # untouched copy for the identity check
    set_attention_mode(backbone)
    if ref is not None: set_attention_mode(ref)
    model = NERModel(backbone, lm_cfg.d_model, NT, HEAD_DROPOUT)
    hp = HP[METHOD]
    if METHOD == "full":
        pass                                                                    # everything trainable
    elif METHOD == "partial":
        n = len(backbone.blocks)
        for p in backbone.parameters(): p.requires_grad = False                 # embeddings + all blocks frozen ...
        for blk in backbone.blocks[n - PARTIAL_K:]:                             # ... except the top K blocks
            for p in blk.parameters(): p.requires_grad = True
        for p in backbone.norm_f.parameters(): p.requires_grad = True           # final RMSNorm feeds the head
        print(f"partial fine-tuning: blocks {n - PARTIAL_K}..{n - 1} ({PARTIAL_K} of {n}) + final norm + head trainable; "
              f"embeddings and blocks 0..{n - PARTIAL_K - 1} frozen")
        if ATTN_MODE == "bidir":
            print("WARNING: ATTN_MODE='bidir' unmasks the FROZEN lower blocks too (they see an attention pattern they were never "
                  "trained for and cannot adapt). 'bidir_top' keeps them exactly as in Phase 1.")
    elif METHOD in ("lora", "lorma"):
        for p in backbone.parameters(): p.requires_grad = False
        if METHOD == "lora":
            n = apply_adapter(backbone, ADAPTER_TARGETS, lambda l: LoRALinear(l, hp["r"], hp["alpha"], hp["dropout"]))
            print(f"LoRA applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
        else:
            n = apply_adapter(backbone, ADAPTER_TARGETS, lambda l: LoRMALinear(l, hp["r"], hp["alpha"], hp["variant"]))
            print(f"LoRMA[{hp['variant']}] applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
        # The adapted model must equal the Phase-1 model at initialisation.
        _ids = torch.randint(5, VOCAB_SIZE, (1, 48)); _m = torch.ones_like(_ids)
        with torch.no_grad():
            a, _ = backbone.eval()(_ids, attention_mask=_m); b, _ = ref.eval()(_ids, attention_mask=_m)
        err = (a - b).abs().max().item()
        print(f"identity-at-init check: max|adapted - Phase-1| = {err:.2e}")
        assert err < 1e-4, "adapter does not start from the Phase-1 function"
        backbone.train(); del ref
    elif METHOD == "norm_only":
        for p in backbone.parameters(): p.requires_grad = False
        n = 0
        for m in backbone.modules():
            if isinstance(m, (RMSNorm, LayerNorm)):
                for p in m.parameters(): p.requires_grad = True; n += p.numel()
        print(f"norm-only: {n:,} gain parameters unfrozen")
    model.to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"params: total {tot/1e6:.2f}M | trainable {tr/1e6:.3f}M ({100*tr/tot:.2f}%)")
    model.n_total, model.n_trainable = tot, tr
    return model


def class_weights(train_docs):
    cnt = np.zeros(NT)
    for d in train_docs: cnt += np.bincount(d["tags"], minlength=NT)
    w = np.ones(NT)
    if CLASS_WEIGHT == "sqrt_inv":
        nz = cnt > 0
        w[nz] = np.clip(np.sqrt(cnt.max() / cnt[nz]), 1.0, MAX_CLASS_W)
    return torch.tensor(w, dtype=torch.float32, device=device), cnt

In [ ]:
# Cell 18 — Inference: windows -> probabilities -> averaged per token -> BIO decode -> spans
@torch.no_grad()
def predict_probs(model, docs, batch_size=16):
    """Per-doc [n_tokens, NT] probabilities, averaged over the windows that cover each token."""
    model.eval()
    jobs = [(di, a, b) for di, d in enumerate(docs) for (a, b) in make_windows(len(d["ids"]), WINDOW, STRIDE)]
    jobs.sort(key=lambda j: j[2] - j[1])
    merger = WindowMerger(docs)
    for i in range(0, len(jobs), batch_size):
        chunk = jobs[i:i + batch_size]
        seqs, offs = [], []
        for di, a, b in chunk:
            s, o, _ = build_seq(docs[di]["ids"][a:b].tolist()); seqs.append(s); offs.append(o)
        ids, mask, _ = pad_batch(seqs)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(torch.from_numpy(ids).to(device), torch.from_numpy(mask).to(device))
        probs = logits.float().softmax(-1).cpu().numpy()
        for k, (di, a, b) in enumerate(chunk):
            merger.add(di, a, b, probs[k, offs[k]:offs[k] + (b - a)])
    return merger.result()

def decode_all(docs, probs, o_scale=1.0):
    out = {}
    for d, p in zip(docs, probs):
        if len(p) == 0: out[d["id"]] = []; continue
        q = p.copy(); q[:, 0] *= o_scale                        # o_scale < 1 -> more entities (higher recall)
        out[d["id"]] = tags_to_spans(d["text"], d["offs"], q.argmax(1))
    return out

def tune_o_scale(docs, probs, gold, grid=O_SCALE_GRID):
    rows = []
    for s in grid:
        sc = strict_scores(gold, decode_all(docs, probs, s)); rows.append((s, sc["macro_f1"]))
    best = max(rows, key=lambda r: r[1])
    print("o_scale -> macro-F1:", " | ".join(f"{s}: {f:.4f}" for s, f in rows), f"\n  best o_scale = {best[0]}")
    return best[0], rows

In [ ]:
# Cell 19 — Training loop (AdamW, linear warm-up + cosine, grad accumulation, fp16/bf16 autocast)
def make_epoch_items(train_docs, rng):
    items = []
    for di, d in enumerate(train_docs):
        for a, b in make_windows(len(d["ids"]), WINDOW, STRIDE):
            if (d["tags"][a:b] != 0).any() or rng.random() < NEG_WINDOW_KEEP:
                items.append((di, a, b))
    rng.shuffle(items)
    return items

def train_ner(model, train_docs, val_docs, val_gold, epochs, max_steps=None):
    hp = HP[METHOD]
    cw, tag_counts = class_weights(train_docs)
    print("class weights (O, then B-/I- per label):", np.round(cw.cpu().numpy(), 1).tolist())
    head_p = [p for n, p in model.named_parameters() if n.startswith("head.")]
    dec, nodec = [], []
    for n, p in model.named_parameters():
        if n.startswith("head.") or not p.requires_grad: continue
        (dec if (p.ndim >= 2 and "tok_emb" not in n) else nodec).append(p)
    groups = [dict(params=head_p, lr=hp["head_lr"], base_lr=hp["head_lr"], weight_decay=0.0)]
    if dec:   groups.append(dict(params=dec,   lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"]))
    if nodec: groups.append(dict(params=nodec, lr=hp["lr"], base_lr=hp["lr"], weight_decay=0.0))
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]

    n_items = len(make_epoch_items(train_docs, random.Random(SEED + 1)))
    steps_per_epoch = math.ceil(math.ceil(n_items / BATCH) / ACCUM)
    total = epochs * steps_per_epoch
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        prog = (s - warm) / max(1, total - warm)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    print(f"{n_items} windows/epoch (after negative sub-sampling) | {steps_per_epoch} optimiser steps/epoch | total {total} steps")

    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, losses, step, done = [], [], 0, False
    best = dict(f1=-1.0, epoch=0, state=None)
    t_train = time.time()
    for ep in range(1, epochs + 1):
        items = make_epoch_items(train_docs, random.Random(SEED + ep))
        model.train(); t0 = time.time(); micro, ep_loss, ep_n = 0, 0.0, 0
        nb = math.ceil(len(items) / BATCH)
        for bi in range(nb):
            chunk = items[bi * BATCH:(bi + 1) * BATCH]
            seqs, labs = [], []
            for di, a, b in chunk:
                d = train_docs[di]
                s, o, l = build_seq(d["ids"][a:b].tolist(), d["tags"][a:b].tolist()); seqs.append(s); labs.append(l)
            ids, mask, lab = pad_batch(seqs, labs)
            ids, mask, lab = (torch.from_numpy(x).to(device) for x in (ids, mask, lab))
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(ids, mask)
            loss = F.cross_entropy(logits.float().view(-1, NT), lab.view(-1), weight=cw, ignore_index=-100)
            scaler.scale(loss / ACCUM).backward()
            micro += 1; lv = loss.item(); losses.append(lv); ep_loss += lv; ep_n += 1
            if micro % ACCUM == 0 or bi == nb - 1:
                m = lr_mult(step)
                for g in opt.param_groups: g["lr"] = g["base_lr"] * m
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
                if step % 25 == 0:
                    print(f"  ep {ep} step {step}/{total} | loss {np.mean(losses[-25*ACCUM:]):.4f} | lr x{m:.3f} | {time.time()-t0:.0f}s")
                if max_steps and step >= max_steps: done = True; break
        rec = dict(epoch=ep, train_loss=ep_loss / max(ep_n, 1), seconds=time.time() - t0, steps=step)
        if val_docs:
            probs = predict_probs(model, val_docs)
            sc = strict_scores(val_gold, decode_all(val_docs, probs, 1.0))
            rec.update(val_macro_f1=sc["macro_f1"], val_micro_f1=sc["micro_f1"])
            if sc["macro_f1"] > best["f1"]:
                best = dict(f1=sc["macro_f1"], epoch=ep,
                            state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
        print(f"epoch {ep}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
        history.append(rec)
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0,
                 first_losses=float(np.mean(losses[:20])) if losses else None, last_losses=float(np.mean(losses[-20:])) if losses else None)
    if best["state"] is None:                                    # no validation: keep the final weights
        best = dict(f1=None, epoch=history[-1]["epoch"],
                    state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
    return history, best, stats

In [ ]:
# Cell 20 — Build data, check the tokenisation ceiling, build the model, train
seed_everything(SEED)
if TRIM_PUNCT:
    TRIM_END, TRIM_START = learn_trim_chars({did: GOLD_ALL[did] for f in TRAIN_FOLDS for did in GOLD[f]}, ALL_TEXTS)
    print("learned edge-trim characters | end:", "".join(sorted(TRIM_END)), "| start:", "".join(sorted(TRIM_START)))
train_docs = build_docs(TRAIN_FOLDS, cache_name="train_" + "_".join(map(str, TRAIN_FOLDS)))
train_gold = {d["id"]: GOLD_ALL[d["id"]] for d in train_docs}
oracle_train = oracle_test(train_docs, train_gold)           # the best ANY model could reach with this tokenisation

val_docs, val_gold = [], {}
if VAL_FOLDS:
    val_docs = build_docs(VAL_FOLDS, cache_name="val_" + "_".join(map(str, VAL_FOLDS)))
    if SMOKE_TEST: val_docs = val_docs[:SMOKE_VAL_DOCS]
    val_gold = {d["id"]: GOLD_ALL[d["id"]] for d in val_docs}          # ONLY the docs we predict on
print(f"train docs {len(train_docs)} | val docs {len(val_docs)}")

model = build_ner_model()
history, best, stats = train_ner(model, train_docs, val_docs, val_gold,
                                 epochs=EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)           # best epoch (or final weights if no validation)
print(f"\nbest epoch {best['epoch']} | val macro-F1 {best['f1']} | train {stats['train_seconds']/60:.1f} min | "
      f"peak GPU {stats['peak_gpu_gb']:.1f} GB")

In [ ]:
# Cell 21 — Smoke-test verdict (SMOKE_TEST=True only)
if SMOKE_TEST and val_docs:
    drop = stats["last_losses"] / stats["first_losses"]
    probs = predict_probs(model, val_docs)
    pred = decode_all(val_docs, probs, 1.0)
    n_pred = sum(len(v) for v in pred.values()); n_gold = sum(len(v) for v in val_gold.values())
    sc = strict_scores(val_gold, pred)
    print(f"loss: first-20 {stats['first_losses']:.3f} -> last-20 {stats['last_losses']:.3f}  (ratio {drop:.2f})")
    print(f"dev subset: predicted {n_pred} spans vs {n_gold} gold | strict macro-F1 {sc['macro_f1']:.4f} | micro-F1 {sc['micro_f1']:.4f}")
    print("tokenisation ceiling on train (oracle) macro-F1:", round(oracle_train["macro_f1"], 4))
    ok = drop < 0.5 and n_pred > 0 and sc["micro_f1"] > 0.1
    print("\nSMOKE VERDICT:", "PASS - the pipeline learns. Now run with SMOKE_TEST=False (keep METHOD='partial')." if ok else
          "CHECK - loss barely moved or (almost) nothing was predicted. Debug the pipeline first "
          "(check the oracle number above, ATTN_MODE, learning rate); METHOD='full' is the most forgiving setting.")
    print("(heuristic: 250 steps is short; a micro-F1 of 0.3-0.6 on the subset is normal. Zero means a bug.)")
else:
    print("not a smoke run")

In [ ]:
# Cell 22 — Dev-fold analysis: tune the O-scale, per-label scores, boundary-vs-label errors (full runs only)
O_SCALE = O_SCALE_FINAL                                      # used as-is when there is no validation fold
dev_scores = None
if not SMOKE_TEST and val_docs:
    probs_val = predict_probs(model, val_docs)
    O_SCALE, _ = tune_o_scale(val_docs, probs_val, val_gold)
    pred_val = decode_all(val_docs, probs_val, O_SCALE)
    dev_scores = strict_scores(val_gold, pred_val)
    print_scores(dev_scores, f"[dev, o_scale={O_SCALE}]")
    print("error breakdown of missed gold spans:", error_breakdown(val_gold, pred_val))
    print("oracle ceiling (train folds):", round(oracle_train["macro_f1"], 4))
else:
    print("skipped (smoke run or no validation fold)")

In [ ]:
# Cell 23 — Test predictions -> prediction.json -> submission.zip (full runs only)
def write_submission(pred_by_doc, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    sub = {}
    for did in TEST_TEXTS:
        L = len(TEST_TEXTS[did]); lst = []
        for s, e, l in pred_by_doc.get(did, []):
            assert isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= L and l in LABEL_SET
            lst.append([int(s), int(e - 1) if END_INCLUSIVE else int(e), l])     # same end-convention as the gold
        sub[did] = lst
    assert set(sub) == set(TEST_TEXTS), "submission ids differ from the official test ids"
    (out_dir / "prediction.json").write_text(json.dumps(sub, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")      # must sit at the ROOT of the zip
    n = sum(len(v) for v in sub.values())
    print(f"wrote {out_dir/'submission.zip'} | {len(sub)} docs | {n} spans | per label:",
          dict(Counter(x[2] for v in sub.values() for x in v).most_common()))
    verify_submission(out_dir / "submission.zip")
    return sub


def verify_submission(zip_path):
    """Re-open the zip exactly like the evaluator would and check everything it checks (ids, structure, spans, labels)."""
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"the zip must contain ONLY prediction.json at its root, found {names}"
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, dict), "prediction.json must be a JSON object {doc_id: [[start, end, label], ...]}"
    missing, unknown = set(TEST_TEXTS) - set(sub), set(sub) - set(TEST_TEXTS)
    assert not missing and not unknown, f"ids differ from the Phase-2 test set: {len(missing)} missing, {len(unknown)} unknown"
    for did, lst in sub.items():
        assert isinstance(lst, list), f"{did}: value must be a LIST of [start, end, label] (a dict means this is not a Phase-2 file)"
        for e in lst:
            assert isinstance(e, list) and len(e) in (3, 4), f"{did}: bad entry {e!r}"
            assert isinstance(e[0], int) and isinstance(e[1], int) and isinstance(e[2], str) and e[2] in LABEL_SET, f"{did}: bad entry {e!r}"
            s_, e_ = e[0], e[1] + (1 if END_INCLUSIVE else 0)                     # back to an exclusive end
            assert 0 <= s_ < e_ <= len(TEST_TEXTS[did]), f"{did}: offsets out of range {e!r}"
    print(f"VERIFIED {zip_path.resolve()} | {len(sub)} documents | first ids {list(sub)[:3]}")
    print("   -> upload exactly THIS file to the Phase-2 Codabench page (not the Phase-1 submission.zip, which has the same name).")

if not SMOKE_TEST:
    test_docs = docs_from_texts(TEST_TEXTS)
    probs_test = predict_probs(model, test_docs)
    pred_test = decode_all(test_docs, probs_test, O_SCALE)
    submission = write_submission(pred_test, RUN_DIR / "submission")
    # spot-check: the text under a few predicted spans should look like the label says
    for did in list(submission)[:2]:
        for s, e, l in submission[did][:4]:
            e_ex = e + 1 if END_INCLUSIVE else e
            print(f"  {did} {l:9s} {TEST_TEXTS[did][s:e_ex]!r}")
else:
    print("smoke run: no submission written")

In [ ]:
# Cell 24 — Save trainable weights + the run log (for the report's compute / memory / PEFT-vs-full tables)
log = dict(run=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, partial_k=PARTIAL_K, bidir_top_k=BIDIR_TOP_K, pos_type=lm_cfg.pos_type, smoke=SMOKE_TEST,
           train_folds=TRAIN_FOLDS, val_folds=VAL_FOLDS, hp=HP[METHOD], epochs=EPOCHS, batch=BATCH, accum=ACCUM,
           window=WINDOW, stride=STRIDE, neg_window_keep=NEG_WINDOW_KEEP, class_weight=CLASS_WEIGHT,
           params_total=model.n_total, params_trainable=model.n_trainable, history=history, best_epoch=best["epoch"],
           stats=stats, o_scale=O_SCALE, oracle_macro_f1=oracle_train["macro_f1"],
           dev_macro_f1=None if dev_scores is None else dev_scores["macro_f1"],
           dev_per_label=None if dev_scores is None else {l: round(v["f1"], 4) for l, v in dev_scores["per"].items()})
(RUN_DIR / "run_log.json").write_text(json.dumps(log, indent=2, default=float))
torch.save({"trainable": best["state"], "method": METHOD, "attn_mode": ATTN_MODE, "tags": TAGS, "log": log},
           RUN_DIR / f"ner_{RUN_NAME}.pt")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))

In [ ]:
# Cell 25 — Export this run's per-token distributions ("soft labels") so it can serve as the KD teacher (run #6)
# Same shared BPE tokenizer in every architecture => token i of a document is the same token for every model, so these arrays can
# teach ANY student without needing this notebook's model code.  Stored as log-probabilities (float16): softmax(logp / T) is
# exactly the temperature-softened distribution softmax(z / T), because logp = z - logsumexp(z).
if EXPORT_SOFT_LABELS and not SMOKE_TEST:
    _t0 = time.time(); _out = {}
    for _f in EXPORT_FOLDS:
        _docs = build_docs([_f], cache_name=f"export_{_f}")
        _probs = predict_probs(model, _docs)                        # raw probabilities, windows averaged (NO O-scale)
        _out[_f] = {d["id"]: np.log(np.maximum(p, 1e-8)).astype(np.float16) for d, p in zip(_docs, _probs)}
        print(f"fold {_f}: {len(_docs)} docs, {sum(len(p) for p in _probs)/1e6:.2f}M tokens")
    _meta = dict(teacher=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, window=WINDOW, stride=STRIDE,
                 params_total=int(model.n_total), params_trainable=int(model.n_trainable),
                 dev_macro_f1=None if dev_scores is None else float(dev_scores["macro_f1"]), o_scale=float(O_SCALE),
                 train_folds=list(TRAIN_FOLDS), val_folds=list(VAL_FOLDS), epochs=int(EPOCHS), best_epoch=int(best["epoch"]))
    _path = RUN_DIR / f"teacher_softlabels_{RUN_NAME}.pt"
    torch.save(dict(meta=_meta, tags=list(TAGS), folds=_out), _path)
    print(f"saved {_path} ({_path.stat().st_size/1e6:.0f} MB) in {time.time()-_t0:.0f}s | teacher params {_meta['params_total']/1e6:.1f}M "
          f"| dev macro-F1 {_meta['dev_macro_f1']}")
else:
    print("soft-label export skipped (EXPORT_SOFT_LABELS=False or smoke run)")

## What to do next

1. **First run (these defaults):** `SMOKE_TEST=True, METHOD="partial"`. Read the *attention:* and *partial fine-tuning:* lines printed in Cell 20 (blocks 0-8 causal / 9-12 bidirectional, trainable params ≈ 11.5M of 53.9M), the **ORACLE** line, and the smoke verdict.
2. **Run #4 of your plan:** `RUN_NAME="dec03_partial"`, `SMOKE_TEST=False`. Trains on fold 1, validates on fold 2, writes `submission/submission.zip`; upload it to Codabench.
3. **Free ablations for the report** (dev only, change one thing): `PARTIAL_K` ∈ {0, 2, 4, 8, 13} (0 = linear probe, 13 = all blocks), `ATTN_MODE` `bidir_top` vs `bidir` vs `causal`, and `partial` vs `full` / `lorma` on this same model.
4. **Keep the artifacts** of every run (`ner_*.pt`, `run_log.json`). The KD run (#6 in your order) needs a teacher: the best dev macro-F1 among your finished runs.
5. **Final fit:** `TRAIN_FOLDS=[1,2]`, `VAL_FOLDS=[]`, `EPOCHS=` best epoch, `O_SCALE_FINAL=` the `o_scale` printed by the dev run.
6. Next in your order: the **KD notebook** (teacher → smaller student).

**Want this run to be the KD teacher (run #6)?** Cell 25 writes `teacher_softlabels_<RUN_NAME>.pt` (per-token log-probabilities for folds 1 and 2). Either set `EXPORT_SOFT_LABELS=True` in Cell 2 before the full run, or, after a finished full run, execute `EXPORT_SOFT_LABELS = True` in a new cell and then run Cell 25 (the trained model is still in memory). Save the output as a Kaggle dataset to use it in the KD notebook.